<h1>📊 Biofilter — Report: <code>platform_data_statistics</code></h1>

What this bundle holds: how much, of what, and how big.

A platform report — it describes the **bundle**, not the biology in it.
Reach for it when you pick up a bundle you did not build and want to know
what is actually in there.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "platform_data_statistics"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

### 2. One row per measurement

Heterogeneous statistics do not fit a wide table, so this one is long:
`section` and `metric` name the measurement, `dimension_1` and
`dimension_2` say what it is measured by.

In [ ]:
result = bf.report.run(REPORT)
stats = result.to_pandas()

print(f"{len(stats)} measurements")
stats.groupby("section").size()

### 3. Which build is this

`bundle_id` is what ties a result back to the data that produced it —
the same id the provenance sidecar carries.

In [ ]:
stats[stats["section"] == "bundle"][
    ["metric", "value_number", "value_text", "note"]
]

`tables_without_rows` is the number to watch. A declared table with no
rows is a source that was planned and did not land — it gets a
measurement of its own rather than hiding in the per-table list.

### 4. How big, per table

Rows, bytes and file count come from `manifest.json`, so this section
costs **no I/O at all** — the sizes of a 21 GB bundle are read from a few
hundred lines of JSON.

In [ ]:
storage = stats[stats["section"] == "storage"].sort_values(
    "value_number", ascending=False
)

storage[["dimension_1", "dimension_2", "value_number", "value_text", "note"]].head(12)

In [ ]:
import time

# The manifest-only sections, timed.
started = time.perf_counter()
bf.report.run(REPORT, sections=["bundle", "storage"])
print(f"bundle + storage: {time.perf_counter() - started:.2f}s")

started = time.perf_counter()
bf.report.run(REPORT)
print(f"everything:       {time.perf_counter() - started:.2f}s")

### 5. What kinds of thing are in here

In [ ]:
stats[stats["section"] == "entities"][["dimension_1", "value_number"]]

### 6. Variants per chromosome

Grouped from the data, not parsed out of filenames: the manifest counts
rows per *file*, and a file happening to be one chromosome is a
convention of the current build rather than a guarantee.

It is affordable because `chromosome` is a real column with row-group
statistics — 2.2 billion rows group in about a second.

In [ ]:
variants = stats[stats["section"] == "variants"]

if len(variants):
    wide = variants.pivot_table(
        index="dimension_2", columns="dimension_1",
        values="value_number", aggfunc="sum",
    )
    wide.index.name = "chromosome"
    display(wide.head(25))
else:
    print("this bundle carries no variant tables")

⚠️ **A bundle built for a subset of chromosomes shows exactly that.**
If only one chromosome appears here, every variant count elsewhere is
about that chromosome — true of the bundle, not of the genome.

### 7. How things are connected

In [ ]:
pairs = stats[
    (stats["section"] == "relationships")
    & (stats["metric"] == "relationships_by_group_pair")
]

pairs[["dimension_1", "dimension_2", "value_number"]].head(10)

In [ ]:
stats[
    (stats["section"] == "relationships")
    & (stats["metric"] == "relationships_by_type")
][["dimension_1", "value_number", "value_text"]]

### 8. What each source contributed

Every data source is listed, including ones that never ran — those have a
null `value_text`. `platform_etl_status` is where to go for why.

In [ ]:
sources = stats[stats["section"] == "sources"]

print(f"{len(sources)} sources; {int(sources['value_text'].isna().sum())} never ran")
sources[["dimension_1", "dimension_2", "value_number", "value_text", "as_of"]].head(10)

### 9. Export

In [ ]:
for path in result.write(OUTPUT_DIR / "platform_data_statistics.csv"):
    print(path)

### 10. The same thing on the command line

```bash
biofilter report run --report-name platform_data_statistics --output stats.csv

# Just the free parts:
biofilter report run --report-name platform_data_statistics \\
    --param sections=bundle --param sections=storage
```